In [1]:
# Path + packaging
import sys  # no installation needed
from pathlib import Path  # no installation needed

SRC_DIR = Path(r"C:\Users\quantbase\Desktop\SyStrat\src")
PKG_DIR = SRC_DIR / "syslib"
PKG_DIR.mkdir(parents=True, exist_ok=True)
(PKG_DIR / "__init__.py").touch(exist_ok=True)  # ensure it's importable

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("sys.path OK:", SRC_DIR in map(Path, map(str, sys.path)))


sys.path OK: True


In [2]:
from pathlib import Path
import pandas as pd, numpy as np
import importlib
import datetime
from datetime import date
from syslib.size_legacy import SizeConfig, size_snapshot_ewma
from syslib.size_ml import size_snapshot_ml
import syslib.size_legacy as sl; importlib.reload(sl)
import syslib.size_ml as sml; importlib.reload(sml)

<module 'syslib.size_ml' from 'C:\\Users\\quantbase\\Desktop\\SyStrat\\src\\syslib\\size_ml.py'>

In [3]:
RUN_DATE = date.today().strftime("%d-%m-%Y") #"08-12-2025"  
BASE = Path(r"C:\Users\quantbase\Desktop\SyStrat") / RUN_DATE
FIGDIR = BASE / "figures"

In [4]:
k = 10

Xy = pd.read_parquet(BASE / "data_int/ml/features_raw_long.parquet")
Xy = Xy.sort_values(["ticker","date"]).reset_index(drop=True)

# 40/35/25 per-ticker time split
rows_test = []
for tkr, g in Xy.groupby("ticker", sort=True):
    n = len(g); i1 = int(np.floor(0.40*n)); i2 = i1 + int(np.floor(0.35*n))
    g_te = g.iloc[i2:].copy()
    rows_test.append(g_te[["date","ticker"]])

rowmap_test = pd.concat(rows_test, ignore_index=True)
rowmap_test.to_parquet(BASE / "data_int/ml/rowmap_test_k10.parquet")
rowmap_test.head(), rowmap_test.tail(), len(rowmap_test)

(        date   ticker
 0 2024-06-22  ADA-USD
 1 2024-06-23  ADA-USD
 2 2024-06-24  ADA-USD
 3 2024-06-25  ADA-USD
 4 2024-06-26  ADA-USD,
            date   ticker
 3693 2025-12-01  XRP-USD
 3694 2025-12-02  XRP-USD
 3695 2025-12-03  XRP-USD
 3696 2025-12-04  XRP-USD
 3697 2025-12-05  XRP-USD,
 3698)

In [5]:
from syslib.size_ml import size_snapshot_ml
from syslib.size_legacy import SizeConfig, size_snapshot_ewma

In [6]:
# Portfolio weights (example; use yours)
weights = {
    "BTC-USD":0.23, "ETH-USD":0.078, "BNB-USD":0.078,
    "XRP-USD":0.078, "ADA-USD":0.078, "LINK-USD":0.078, "SOL-USD":0.078
}
mult = {t: 1.0 for t in weights}

In [7]:
cfg = SizeConfig(
    notional=50_000.0, leverage=2.5, risk_fraction=0.02,
    window_len=32, w0=0.6, decay=0.4, hist_span=54, blend_recent=0.7, scale=19.0,
    lookback_years=5, interval="1d", price_mode="prev_close",
)

In [8]:
#check 'k'
provider_kwargs = {
    "base_dir": BASE, "k_label": 10, "model_tag": "xgb",
    "rowmap_path": BASE / "data_int/ml/rowmap_test_k10.parquet",  # << use this CHECK LOCATION (k)
    # if your rowmap ticker column isn’t 'ticker', set:
    # "rowmap_ticker_col": "Ticker",
    # to pick EN instead of GBM:
    # "pred_model_col": "pred_en",
}

In [9]:
# 3) EWMA snapshot
snap_ewma = size_snapshot_ewma(weights, mult_map=mult, cfg=cfg, prices=None)
snap_ewma.to_csv(BASE / "holdings_ewma_snapshot.csv", float_format="%.8f")
display(snap_ewma)

,portw,mult,last_price,pred_vol,units,notional_alloc
ticker,,,,,,
ADA-USD,0.078,1.0,0.396584,0.351869,1397.392296,554.183432
BNB-USD,0.078,1.0,877.806885,0.266414,0.833833,731.944385
BTC-USD,0.230,1.0,88175.179688,0.193480,0.033704,2971.881896
ETH-USD,0.078,1.0,3060.594727,0.334396,0.190532,583.140963
LINK-USD,0.078,1.0,13.292393,0.341377,42.973153,571.216032
SOL-USD,0.078,1.0,129.481094,0.297678,5.059201,655.070897
XRP-USD,0.078,1.0,1.979287,0.358497,274.814781,543.937332


In [10]:
cfg = SizeConfig(
    notional=50_000.0, leverage=2.5, risk_fraction=0.02/19,
    window_len=32, w0=0.6, decay=0.4, hist_span=54, blend_recent=0.7, scale=19.0,
    lookback_years=5, interval="1d", price_mode="prev_close",
)

In [11]:
snap_ml = size_snapshot_ml(weights, provider_kwargs, mult_map=mult, cfg=cfg, prices=None)
snap_ml.to_csv(BASE / "holdings_ml_snapshot.csv", float_format="%.8f")
snap_ml 

,portw,mult,last_price,pred_vol,units,notional_alloc
ticker,,,,,,
ADA-USD,0.078,1.0,0.396584,0.037101,697.527982,276.628440
BNB-USD,0.078,1.0,877.806885,0.031146,0.375391,329.520869
BTC-USD,0.230,1.0,88175.179688,0.020263,0.016938,1493.530115
ETH-USD,0.078,1.0,3060.594727,0.035774,0.093737,286.891611
LINK-USD,0.078,1.0,13.292393,0.039614,19.490627,259.077066
SOL-USD,0.078,1.0,129.481094,0.037854,2.093928,271.124119
XRP-USD,0.078,1.0,1.979287,0.035123,147.631644,292.205397


In [12]:
float(snap_ewma['notional_alloc'].sum()), float(snap_ml['notional_alloc'].sum())

(6611.374936672976, 3208.9776172025163)

In [13]:
# 5) Side-by-side comparison (units, last_price, pred_vol)
cmp = (
    snap_ewma[["units","last_price","pred_vol","notional_alloc"]].rename(columns={"units":"units_ewma","pred_vol":"pred_vol_ewma","notional_alloc":"notional_alloc"})
    .join(
        snap_ml[["units","last_price","pred_vol","notional_alloc"]].rename(columns={"units":"units_ewma","pred_vol":"pred_vol_ewma","notional_alloc":"notional_alloc"}),
        how="outer", lsuffix="_ewma", rsuffix="_ml"
    )
)
cmp.to_csv(BASE / "holdings_snapshot_compare.csv", float_format="%.8f")
display(cmp)

,units_ewma_ewma,last_price_ewma,pred_vol_ewma_ewma,notional_alloc_ewma,units_ewma_ml,last_price_ml,pred_vol_ewma_ml,notional_alloc_ml
ticker,,,,,,,,
ADA-USD,1397.392296,0.396584,0.351869,554.183432,697.527982,0.396584,0.037101,276.628440
BNB-USD,0.833833,877.806885,0.266414,731.944385,0.375391,877.806885,0.031146,329.520869
BTC-USD,0.033704,88175.179688,0.193480,2971.881896,0.016938,88175.179688,0.020263,1493.530115
ETH-USD,0.190532,3060.594727,0.334396,583.140963,0.093737,3060.594727,0.035774,286.891611
LINK-USD,42.973153,13.292393,0.341377,571.216032,19.490627,13.292393,0.039614,259.077066
SOL-USD,5.059201,129.481094,0.297678,655.070897,2.093928,129.481094,0.037854,271.124119
XRP-USD,274.814781,1.979287,0.358497,543.937332,147.631644,1.979287,0.035123,292.205397


In [14]:
#----------------Misc---------------

In [26]:
# EWMA snapshot (as before)
#snap_ewma = size_snapshot_ewma(weights, mult_map=mult, cfg=cfg, prices=None)

In [27]:
# ML snapshot (as before)
#snap_ml   = size_snapshot_ml(weights, provider_kwargs, mult_map=mult, cfg=cfg, prices=None)

In [25]:
float(snap_ewma['notional_alloc'].sum()), float(snap_ml['notional_alloc'].sum())

(4171.262749757247, 5296.442024729115)

In [24]:
# Vol level cross-check (medians should be similar order now)
snap_ewma['pred_vol'].median(), snap_ml['pred_vol'].median()*10

(0.5515985428796919, 0.23526519536972046)